In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
EVAL_DIR = DATA_DIR / "evaluation"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation dir: {EVAL_DIR}")

Project root: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai
Evaluation dir: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/evaluation


In [2]:
import json
from pathlib import Path

import pandas as pd

from app.services.rag_service import RAGService
from app.services.retriever_service import RetrieverService
from app.services.vectorstore_service import VectorStoreService
from app.services.embedding_service import EmbeddingService

In [3]:
test_set_path = EVAL_DIR / "test_questions.json"

with test_set_path.open("r", encoding="utf-8") as f:
    test_set = json.load(f)

test_set

[{'id': 'q1',
  'question': 'Was ist Pflegegrad 3?',
  'expected_answer': 'Schwere Beeinträchtigungen der Selbständigkeit oder der Fähigkeiten; Zuordnung bei 47,5 bis unter 70 Gesamtpunkten.',
  'expected_topics': ['pflegeversicherung', 'pflegegrad'],
  'expected_sources': ['SGB_11.pdf']},
 {'id': 'q2',
  'question': 'Wer hat Anspruch auf Pflegegeld?',
  'expected_answer': 'Versicherte mit Pflegegrad 2 bis 5 bei häuslicher Pflege durch geeignete Pflegepersonen.',
  'expected_topics': ['pflegegeld', 'pflegeversicherung'],
  'expected_sources': ['SGB_11.pdf']},
 {'id': 'q3',
  'question': 'Wie hoch ist der Entlastungsbetrag?',
  'expected_answer': 'Der Entlastungsbetrag beträgt monatlich 125 Euro.',
  'expected_topics': ['entlastungsbetrag', 'pflegeleistungen'],
  'expected_sources': ['SGB_11.pdf']},
 {'id': 'q4',
  'question': 'Was regelt das PUEG?',
  'expected_answer': 'Änderungen zur Unterstützung und Entlastung in der Pflege, inklusive Leistungsanpassungen und Strukturreformen.',
  

In [4]:
vectorstore_service = VectorStoreService()
retriever = RetrieverService(vectorstore_service)
rag = RAGService(retriever_service=retriever)

/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/app/services/vectorstore_service.py:26: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._store = Chroma(


In [5]:
records = []

for item in test_set:
    question = item["question"]
    retrieved = retriever.retrieve(question, top_k=5)

    records.append(
        {
            "id": item["id"],
            "question": question,
            "expected_answer": item["expected_answer"],
            "expected_sources": ", ".join(item["expected_sources"]),
            "retrieved_sources": ", ".join(
                sorted({chunk.metadata.extra.get("filename", Path(chunk.source_path).name) for chunk in retrieved})
            ),
            "retrieved_chunk_ids": ", ".join(chunk.chunk_id for chunk in retrieved),
            "retrieved_preview": retrieved[0].text[:400] if retrieved else "",
            "hit": any(
                Path(chunk.source_path).name in item["expected_sources"]
                for chunk in retrieved
            ),
        }
    )

df_eval = pd.DataFrame(records)
df_eval

,id,question,expected_answer,expected_sources,retrieved_sources,retrieved_chunk_ids,retrieved_preview,hit
0,q1,Was ist Pflegegrad 3?,Schwere Beeinträchtigungen der Selbständigkeit...,SGB_11.pdf,SGB_11.pdf,0b859b80-8f28-4172-978c-feb589417aa0-chunk-088...,ach\nMaßgabe von Satz 3 einem Pflegegrad zugeo...,True
1,q2,Wer hat Anspruch auf Pflegegeld?,Versicherte mit Pflegegrad 2 bis 5 bei häuslic...,SGB_11.pdf,"SGB_11.pdf, SGB_5.pdf",0b859b80-8f28-4172-978c-feb589417aa0-chunk-024...,ige der Pflegegrade 2 bis 5 können anstelle de...,True
2,q3,Wie hoch ist der Entlastungsbetrag?,Der Entlastungsbetrag beträgt monatlich 125 Euro.,SGB_11.pdf,"PUEG_RefE_Pflegereform_bf.pdf, SGB_11.pdf",0b859b80-8f28-4172-978c-feb589417aa0-chunk-088...,"nuar 2017 zustehen, nicht um jeweils mindesten...",True
3,q4,Was regelt das PUEG?,Änderungen zur Unterstützung und Entlastung in...,PUEG_RefE_Pflegereform_bf.pdf,"PUEG_RefE_Pflegereform_bf.pdf, SGB_11.pdf, SGB...",a045aad7-0d31-46df-8243-289cb921641a-chunk-000...,Referentenentwurf\ndes Bundesministeriums für ...,True
4,q5,Welche Leistungen sind in der gesetzlichen Kra...,Leistungen der gesetzlichen Krankenversicherun...,SGB_5.pdf,SGB_5.pdf,676aebc2-2db0-46c9-9086-72ed95b36887-chunk-001...,"durch Aufklärung, Beratung und Leistungen zu\n...",True


In [6]:
df_eval[[
    "id",
    "question",
    "expected_sources",
    "retrieved_sources",
    "hit",
]]

,id,question,expected_sources,retrieved_sources,hit
0,q1,Was ist Pflegegrad 3?,SGB_11.pdf,SGB_11.pdf,True
1,q2,Wer hat Anspruch auf Pflegegeld?,SGB_11.pdf,"SGB_11.pdf, SGB_5.pdf",True
2,q3,Wie hoch ist der Entlastungsbetrag?,SGB_11.pdf,"PUEG_RefE_Pflegereform_bf.pdf, SGB_11.pdf",True
3,q4,Was regelt das PUEG?,PUEG_RefE_Pflegereform_bf.pdf,"PUEG_RefE_Pflegereform_bf.pdf, SGB_11.pdf, SGB...",True
4,q5,Welche Leistungen sind in der gesetzlichen Kra...,SGB_5.pdf,SGB_5.pdf,True


In [7]:
hit_rate = df_eval["hit"].mean()
print(f"Hit rate@5: {hit_rate:.2%}")

Hit rate@5: 100.00%


In [8]:
for item in test_set:
    result = rag.answer(item["question"], top_k=5)
    print("=" * 100)
    print(item["question"])
    print(result.answer)
    print("Sources:")
    for src in result.sources:
        print(src)
    print()

Was ist Pflegegrad 3?
Pflegegrad 3 bedeutet schwere Beeinträchtigungen der Selbständigkeit oder der Fähigkeiten. Er wird bei Gesamtpunkten von 47,5 bis unter 70 vergeben. (Quelle: § 15 Absatz 3, Begutachtungsinstrument – 0b859b80-8f28-4172-978c-feb589417aa0, chunk 131)
Sources:
document_id='0b859b80-8f28-4172-978c-feb589417aa0' chunk_id='0b859b80-8f28-4172-978c-feb589417aa0-chunk-0880' source='gesetze-im-internet' score=None
document_id='0b859b80-8f28-4172-978c-feb589417aa0' chunk_id='0b859b80-8f28-4172-978c-feb589417aa0-chunk-0130' source='gesetze-im-internet' score=None
document_id='0b859b80-8f28-4172-978c-feb589417aa0' chunk_id='0b859b80-8f28-4172-978c-feb589417aa0-chunk-0307' source='gesetze-im-internet' score=None
document_id='0b859b80-8f28-4172-978c-feb589417aa0' chunk_id='0b859b80-8f28-4172-978c-feb589417aa0-chunk-0131' source='gesetze-im-internet' score=None
document_id='0b859b80-8f28-4172-978c-feb589417aa0' chunk_id='0b859b80-8f28-4172-978c-feb589417aa0-chunk-0127' source='ges

In [9]:
output_path = PROCESSED_DIR / "retrieval_evaluation_results.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_eval.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed/retrieval_evaluation_results.csv
